In [ ]:
# Langchain Chatbot using Redis

In [ ]:
%pip install -U langchain-redis
%pip install -U langchain-core
%pip install -U langchain-community

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------- ----------------- 1.3/2.4 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------------ --------------- 1.3/2.2 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.1 MB/s  0:00:00

   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 

In [6]:
from dotenv import load_dotenv # to connect with env file
from langchain_google_genai import ChatGoogleGenerativeAI # to connect with llm
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.chat_message_histories import RedisChatMessageHistory

C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_5152\3152012572.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import RedisChatMessageHistory


In [ ]:
# langchain considers everything as a tool so to make our function tool and be accessible by langchain with use @annotation in front of function to make it a tool for langchain
from langchain.tools import tool
# to use langchain as an agent 
from langchain.agents import create_agent

# When we create an agent we will first pass its version and its temperature
# After creating an agent firstly we pass this llm model and will pass this info stored in it
# The 2nd thing we will pass is tools (@tool which we wrote above function, will pass that function in this tool)
# The 3rd thing we will pass is our prompt
#The tool will store data in list as there might be more than 1 tools working together

In [7]:
load_dotenv()

True

In [8]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash",
    temperature = 0
)

In [11]:
# messages = []
# as we are storing conversation in redis so we will not use the above list now and store it in below redis session.
history = RedisChatMessageHistory(
    session_id = "student_1",
    url = "redis://localhost:6379"
)

In [13]:
question = input("You :")

human_message = HumanMessage(
    content=question
)

history.add_message(human_message)

response = llm.invoke(history.messages)

ai_message = AIMessage(
    content=response.content
)

history.add_message(ai_message)

print("AI : ",ai_message.content[0]["text"])

AI :  Your name is **Priyanshu**. 

Now, let's dive into how to implement a chatbot that uses a **Redis server** to store and manage chat history.

---

### Why use Redis for Chatbot History?
Redis is an in-memory data structure store. It is incredibly fast, making it perfect for:
1. **Session Management:** Storing active chat sessions.
2. **Low Latency:** Retrieving past messages instantly to feed into the LLM (Large Language Model) for context.
3. **TTL (Time-to-Live):** Automatically deleting old chat histories after a period of inactivity (e.g., 24 hours) to save memory.

---

### The Architecture Flow
1. **User** sends a message with a unique `session_id` (e.g., `session_priyanshu`).
2. **Backend Application** fetches the existing chat history for this `session_id` from **Redis**.
3. **Backend** appends the new user message to this history.
4. **Backend** sends the entire history + new message to the **LLM** (like OpenAI GPT).
5. **LLM** generates a response.
6. **Backend** saves 

In [14]:
from langchain_core.messages import get_buffer_string
print(get_buffer_string(history.messages))

Human: My name is priyanshu 
AI: Hello Priyanshu! It's nice to meet you. How can I help you today?
Human: what is my name? also tell me the whole process to implement this chatbot with redis server to store the chat 
AI: Your name is **Priyanshu**. 

Now, let's dive into how to implement a chatbot that uses a **Redis server** to store and manage chat history.

---

### Why use Redis for Chatbot History?
Redis is an in-memory data structure store. It is incredibly fast, making it perfect for:
1. **Session Management:** Storing active chat sessions.
2. **Low Latency:** Retrieving past messages instantly to feed into the LLM (Large Language Model) for context.
3. **TTL (Time-to-Live):** Automatically deleting old chat histories after a period of inactivity (e.g., 24 hours) to save memory.

---

### The Architecture Flow
1. **User** sends a message with a unique `session_id` (e.g., `session_priyanshu`).
2. **Backend Application** fetches the existing chat history for this `session_id` from